## REQUIRES ENV FILE WITH AGOL CREDENTIALS

In [21]:
from typing import List, Sequence, Tuple, Literal
from math import hypot

from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    MultiLineString as ShapelyMultiLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon as ShapelyMultiPolygon,
)
from shapely.ops import transform
import pyproj


from typing import List, Sequence, Tuple, Literal
from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    MultiLineString as ShapelyMultiLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon as ShapelyMultiPolygon,
)
from shapely.ops import transform, unary_union
import pyproj


from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon,
)
from shapely.ops import transform, unary_union, polygonize
import pyproj
import json
import requests
import math
import streamlit as st
import logging
from shapely.geometry import LineString
from shapely.ops import unary_union, linemerge
from typing import Tuple, Optional, List, Dict, Any

In [22]:
def create_buffers(
    geometry_list,
    geom_type,
    distance_m,
    *,
    crs_in="EPSG:4326",
    crs_projected="EPSG:3338",
    crs_out="EPSG:4326",
    cap_style="round",
    join_style="round",
    resolution=16,
):
    """
    Create buffers for points, lines, or polygons and return a
    unified polygon footprint consistent across geometry types.
    """
    if not geometry_list:
        raise ValueError("geometry_list must not be empty")

    # CRS transforms
    to_projected = pyproj.Transformer.from_crs(
        crs_in, crs_projected, always_xy=True
    ).transform
    to_geographic = pyproj.Transformer.from_crs(
        crs_projected, crs_out, always_xy=True
    ).transform

    # Normalize geometries
    shapely_geoms = []
    gt = (geom_type or "").lower()

    for g in geometry_list:
        if isinstance(g, (ShapelyPoint, ShapelyLineString, ShapelyPolygon)):
            shp = g
        else:
            if gt == "point":
                shp = ShapelyPoint(g)

            elif gt in ("line", "linestring"):
                shp = ShapelyLineString(g)

            elif gt == "polygon":
                # Accept either:
                #   - a single ring: [[x,y], [x,y], ...]
                #   - rings wrapper: [ [[x,y], ...] ]  (ESRI-like)
                ring = g
                if (
                    isinstance(g, (list, tuple))
                    and len(g) > 0
                    and isinstance(g[0], (list, tuple))
                    and len(g[0]) > 0
                    and isinstance(g[0][0], (list, tuple))
                ):
                    ring = g[0]

                shp = ShapelyPolygon(ring)

            else:
                raise ValueError(f"Unsupported geometry type: {geom_type}")

        # IMPORTANT: always append, including when input is already a Shapely geometry
        shapely_geoms.append(shp)

    # Buffer each geometry independently (in projected CRS)
    cap_map = {"round": 1, "flat": 2, "square": 3}
    join_map = {"round": 1, "mitre": 2, "miter": 2, "bevel": 3}

    if cap_style not in cap_map:
        raise ValueError(f"Unsupported cap_style: {cap_style}")
    if join_style not in join_map:
        raise ValueError(f"Unsupported join_style: {join_style}")

    buffered = []
    for shp in shapely_geoms:
        projected = transform(to_projected, shp)
        buf = projected.buffer(
            distance_m,
            resolution=resolution,
            cap_style=cap_map[cap_style],
            join_style=join_map[join_style],
        )
        if not buf.is_empty:
            buffered.append(buf)

    if not buffered:
        raise ValueError("Buffering produced no geometry (all buffers were empty)")

    # Dissolve buffers (unified footprint)
    merged = unary_union(buffered)

    # Reproject back to lon/lat
    merged = transform(to_geographic, merged)

    # Enforce validity AFTER reprojection
    if merged.is_empty:
        raise ValueError("Buffered geometry resulted in an empty shape after reprojection")

    if not merged.is_valid:
        # Standard Shapely fix for minor self-intersections, ring issues, etc.
        merged = merged.buffer(0)

    if merged.is_empty or not merged.is_valid:
        raise ValueError("Buffered geometry is invalid after normalization/repair")

    # Extract exterior rings ONLY (ESRI polygon rings)
    polygons = []
    if isinstance(merged, MultiPolygon):
        polygons = list(merged.geoms)
    elif isinstance(merged, ShapelyPolygon):
        polygons = [merged]
    else:
        # Handle GeometryCollection or other polygonal containers defensively
        if hasattr(merged, "geoms"):
            polygons = [g for g in merged.geoms if isinstance(g, ShapelyPolygon)]

    if not polygons:
        raise ValueError(f"Buffered result is not polygonal (type={type(merged).__name__})")

    rings = []
    for poly in polygons:
        if poly.is_empty:
            continue

        coords = list(poly.exterior.coords)
        # ESRI ring must have at least 4 points (closed ring; first==last)
        if len(coords) < 4:
            continue

        rings.append([[float(x), float(y)] for x, y in coords])

    if not rings:
        raise ValueError("No valid exterior rings could be extracted from buffered polygons")

    return rings

In [23]:
import os
import requests
from dotenv import load_dotenv

# Load .env file once at app startup (safe to call multiple times)
load_dotenv()


def get_agol_token() -> str:
    """
    Generates an authentication token for ArcGIS Online using a username and password
    stored in environment variables.

    Environment variables required:
        - AGOL_USERNAME
        - AGOL_PASSWORD

    Returns:
        str: A valid authentication token used to make authorized API requests.

    Raises:
        ValueError: If credentials are missing or authentication fails.
        ConnectionError: If requests cannot reach the AGOL endpoint.
    """
    username = os.getenv("AGOL_USERNAME")
    password = os.getenv("AGOL_PASSWORD")

    if not username or not password:
        raise ValueError("Missing AGOL_USERNAME or AGOL_PASSWORD in environment variables.")

    url = "https://www.arcgis.com/sharing/rest/generateToken"

    data = {
        "username": username,
        "password": password,
        "referer": "https://www.arcgis.com",
        "f": "json"
    }

    try:
        response = requests.post(url, data=data)

        if response.status_code != 200:
            raise Exception(
                f"Request failed with status code {response.status_code}: {response.text}"
            )

        token_data = response.json()

        if "token" in token_data:
            return token_data["token"]
        elif "error" in token_data:
            raise ValueError(
                f"Authentication failed: {token_data['error']['message']}"
            )
        else:
            raise ValueError("Unexpected response format: Token not found.")

    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Failed to connect to ArcGIS Online: {e}")


In [24]:
def query_record(url: str, layer: int, where: str, fields="*", return_geometry=False):
    """
    Executes an SQL-style query against an ArcGIS REST API layer and returns matching records.

    Parameters:
        url: str
            FeatureServer base URL (may or may not already include a layer segment).
        layer: int
            Layer index when url is a FeatureServer root.
        where: str
            SQL-like filter clause (e.g., "GlobalID='...'" or "1=1").
        fields: str
            outFields string. "*" requests all fields.
        return_geometry: bool
            Whether to return geometry in results.

    Returns:
        list: List of 'features' from the ArcGIS REST response.
    """
    token = get_agol_token()
    if not token:
        raise ValueError("Authentication failed: Invalid token.")


    # If the URL already ends with the layer number, don't add it again
    if url.split("/")[-1].isdigit():
        query_url = f"{url}/query"
    else:
        query_url = f"{url}/{layer}/query"

    params = {
        "where": where,
        "outFields": fields,
        "returnGeometry": str(return_geometry).lower(),
        "outSR": 4326,
        "f": "json",
        "token": token
    }

    response = requests.get(query_url, params=params)
    if response.status_code != 200:
        raise Exception(
            f"Request failed with status code {response.status_code}: {response.text}"
        )

    data = response.json()
    if "error" in data:
        raise Exception(
            f"API Error: {data['error']['message']} - {data['error'].get('details', [])}"
        )

    return data.get("features", [])

In [25]:
token = get_agol_token()

apex_url = "https://services.arcgis.com/r4A0V7UzH9fcLVvv/arcgis/rest/services/APEX_PROJECTS_LOADER_APPLICATION/FeatureServer"

# Layer indices used by loaders and query helpers
apex_layers = {
        "projects_layer": 0,
        "locations_layer": 1,
        "sites_layer": 2,
        "routes_layer": 3,
        "boundaries_layer": 4,
        "impact_comms_layer": 5,
        "region_layer": 6,
        "bor_layer": 7,
        "senate_layer": 8,
        "house_layer": 9
        }


In [26]:
projects = query_record(
    apex_url,
    apex_layers['projects_layer'],
    '1=1',
    return_geometry=True
)

project_geometries = []

for project in projects:
    # ---- Reset everything per project ----
    sites = []
    routes = []
    boundaries = []
    geometries = []

    geometry_type = None
    geometry_source = None

    project_attrs = project['attributes']
    project_objectid = project_attrs['objectid']
    project_globalid = project_attrs['globalid']

    # ---- Query child records ----
    sites = query_record(
        url=apex_url,
        layer=apex_layers['sites_layer'],
        where=f"parentglobalid = '{project_globalid}'",
        fields="*",
        return_geometry=True
    )

    routes = query_record(
        url=apex_url,
        layer=apex_layers['routes_layer'],
        where=f"parentglobalid = '{project_globalid}'",
        fields="*",
        return_geometry=True
    )

    boundaries = query_record(
        url=apex_url,
        layer=apex_layers['boundaries_layer'],
        where=f"parentglobalid = '{project_globalid}'",
        fields="*",
        return_geometry=True
    )

    # ---- Identify which collection has records ----
    if len(sites) > 0:
        source_records = sites
        geometry_source = "sites"

    elif len(routes) > 0:
        source_records = routes
        geometry_source = "routes"

    elif len(boundaries) > 0:
        source_records = boundaries
        geometry_source = "boundaries"

    else:
        continue  # No geometry for this project

    # ---- Collect all geometries + determine geometry type ----
    for record in source_records:
        geom = record.get("geometry")
        if geom is None:
            continue

        geometries.append(geom)

        # Determine geometry type once
        if geometry_type is None:
            if "x" in geom and "y" in geom:
                geometry_type = "point"
            elif "paths" in geom:
                geometry_type = "line"
            elif "rings" in geom:
                geometry_type = "polygon"

    # ---- Save project geometry info ----
    if geometries:
        project_geometries.append({
            "project_objectid": project_objectid,
            "project_globalid": project_globalid,
            "geometry_source": geometry_source,
            "geometry_type": geometry_type,
            "geometries": geometries
        })

In [27]:
buffer_inputs = []

for entry in project_geometries:
    project_objectid = entry["project_objectid"]
    project_globalid = entry["project_globalid"]
    geometry_type = entry["geometry_type"]
    geometry_source = entry["geometry_source"]
    
    formatted_geometries = []

    for geom in entry["geometries"]:

        # ---- POINTS ----
        if geometry_type == "point":
            if "x" in geom and "y" in geom:
                formatted_geometries.append([geom["x"], geom["y"]])

        # ---- LINES ----
        elif geometry_type == "line":
            paths = geom.get("paths", [])
            for path in paths:
                line = []
                for coord in path:
                    line.append([coord[0], coord[1]])
                if line:
                    formatted_geometries.append(line)

        # ---- POLYGONS ----
        elif geometry_type == "polygon":
            rings = geom.get("rings", [])
            for ring in rings:
                polygon = []
                for coord in ring:
                    polygon.append([coord[0], coord[1]])
                if polygon:
                    formatted_geometries.append(polygon)

    buffer_inputs.append({
        "project_objectid": project_objectid,
        "project_globalid": project_globalid,
        "geometry_source": geometry_source,
        "geometry_type": geometry_type,
        "coordinates": formatted_geometries
    })

In [28]:
import requests
import json


def upload_geometry_update(
    url,
    layer,
    payload,
    token
):
    """
    Upload geometry updates to an AGOL Feature Layer using applyEdits.
    """
    apply_edits_url = f"{url}/{layer}/applyEdits"

    params = {
        "f": "json",
        "token": token,
        "updates": json.dumps(payload["updates"])
    }

    try:
        response = requests.post(apply_edits_url, data=params, timeout=60)
        response.raise_for_status()
        result = response.json()

        if "updateResults" not in result:
            print("❌ Unexpected response format:")
            print(result)
            return False

        update_result = result["updateResults"][0]

        if update_result.get("success"):
            print(
                f"✅ SUCCESS — ObjectID {update_result.get('objectId')} updated"
            )
            return True

        else:
            error = update_result.get("error", {})
            print(
                f"❌ UPDATE FAILED — ObjectID {update_result.get('objectId')}"
            )
            print(
                f"   Code: {error.get('code')} | Message: {error.get('description')}"
            )
            return False

    except requests.exceptions.RequestException as e:
        print("❌ HTTP REQUEST FAILED")
        print(str(e))
        return False

    except Exception as e:
        print("❌ UNEXPECTED ERROR")
        print(str(e))
        return False


In [29]:
for input in buffer_inputs:
    
    buffer = create_buffers(input['coordinates'], input['geometry_type'], distance_m=.01)

    # --- ESRI Polygon geometry (multipart via rings) ---
    esri_polygon = {
        "rings": buffer,  # list of rings (each is [[lon, lat], ...])
        "spatialReference": {"wkid": 4326},
    }

    # Build payload with .get() and default None
    payload = {
        "updates": [
            {
                "attributes": {
                    "objectid": input['project_objectid'],
                },
                "geometry": esri_polygon
            }
        ]
    }

    upload_geometry_update(
        url=apex_url,
        layer = apex_layers['projects_layer'],
        payload = payload,
        token = token
    )

✅ SUCCESS — ObjectID 91 updated
✅ SUCCESS — ObjectID 92 updated
✅ SUCCESS — ObjectID 93 updated
✅ SUCCESS — ObjectID 94 updated
✅ SUCCESS — ObjectID 95 updated
✅ SUCCESS — ObjectID 96 updated
✅ SUCCESS — ObjectID 97 updated
✅ SUCCESS — ObjectID 98 updated
✅ SUCCESS — ObjectID 99 updated
✅ SUCCESS — ObjectID 100 updated
✅ SUCCESS — ObjectID 101 updated
✅ SUCCESS — ObjectID 102 updated
✅ SUCCESS — ObjectID 103 updated
✅ SUCCESS — ObjectID 104 updated
✅ SUCCESS — ObjectID 105 updated
✅ SUCCESS — ObjectID 106 updated
✅ SUCCESS — ObjectID 107 updated
✅ SUCCESS — ObjectID 108 updated
✅ SUCCESS — ObjectID 109 updated
✅ SUCCESS — ObjectID 110 updated
✅ SUCCESS — ObjectID 111 updated
✅ SUCCESS — ObjectID 112 updated
✅ SUCCESS — ObjectID 113 updated
✅ SUCCESS — ObjectID 114 updated
✅ SUCCESS — ObjectID 115 updated
✅ SUCCESS — ObjectID 116 updated
✅ SUCCESS — ObjectID 117 updated
✅ SUCCESS — ObjectID 118 updated
✅ SUCCESS — ObjectID 119 updated
✅ SUCCESS — ObjectID 120 updated
✅ SUCCESS — ObjectI

In [33]:
buffer_inputs[3]

{'project_objectid': 94,
 'project_globalid': '79c29661-41d1-4ba6-b85b-a684ce06c472',
 'geometry_source': 'routes',
 'geometry_type': 'line',
 'coordinates': [[[-145.468273343816, 62.7789190979181],
   [-145.468275956642, 62.7789257444681],
   [-145.468304131486, 62.7789968072771],
   [-145.468332161149, 62.7790679355042],
   [-145.468360234401, 62.7791391619053],
   [-145.468388589587, 62.779209939417],
   [-145.468417378831, 62.7792813378211],
   [-145.46844574679, 62.7793519176092],
   [-145.468473912719, 62.7794230844746],
   [-145.468501932217, 62.7794942932309],
   [-145.46852977564, 62.7795654807938],
   [-145.468557651519, 62.7796365962869],
   [-145.468585620977, 62.779707507996],
   [-145.468614022844, 62.7797791162736],
   [-145.468642526468, 62.7798501008842],
   [-145.46867116452, 62.7799208913301],
   [-145.468700431529, 62.7799920904199],
   [-145.468729892914, 62.7800627163944],
   [-145.468760112079, 62.7801342186452],
   [-145.468790169685, 62.7802046848268],
   [-145